In [1]:
csv_content = """transaction_id,date,customer_name,product,quantity,unit_price,region
T001,2024-06-01,alice smith,Widget A,3,25.00,east
T002,2024-06-01,Bob Jones,Widget B,1,50.00,West
T003,2024-06-02,alice smith,Widget A,2,25.00,EAST
T004,2024-06-02,Charlie Lee,,5,10.00,south
T005,2024-06-03,Bob Jones,Widget C,-1,30.00,West
T006,2024-06-03,alice smith,Widget A,3,25.00,east
T007,2024-06-04,,Widget B,2,50.00,North
T008,2024-06-04,Diana Ross,Widget A,0,25.00,east
T009,2024-06-05,Bob Jones,Widget B,1,50.00,West
T010,2024-06-05,Bob Jones,Widget B,1,50.00,West"""

with open ("raw_transactions.csv", "w", newline = "") as file:
    file.write(csv_content)


In [2]:
import pandas as pd

In [3]:
def extract(file_path):
    
    print("=" * 50)
    print("EXTRACT PHASE (Bronze Layer)")
    print("=" * 50)
 
    # Read the raw CSV
    df = pd.read_csv(file_path)
 
    # Print extraction summary
    print(f"Source file: {file_path}")
    print(f"Rows extracted: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(f"\nColumn names: {list(df.columns)}")
    print(f"\nData types:\n{df.dtypes}")
    print(f"\nMissing values:\n{df.isnull().sum()}")
    print(f"\nFirst 3 rows:")
    print(df.head(3))
 
    return df

In [4]:
raw_df = extract("raw_transactions.csv")

EXTRACT PHASE (Bronze Layer)
Source file: raw_transactions.csv
Rows extracted: 10
Columns: 7

Column names: ['transaction_id', 'date', 'customer_name', 'product', 'quantity', 'unit_price', 'region']

Data types:
transaction_id     object
date               object
customer_name      object
product            object
quantity            int64
unit_price        float64
region             object
dtype: object

Missing values:
transaction_id    0
date              0
customer_name     1
product           1
quantity          0
unit_price        0
region            0
dtype: int64

First 3 rows:
  transaction_id        date customer_name   product  quantity  unit_price  \
0           T001  2024-06-01   alice smith  Widget A         3        25.0   
1           T002  2024-06-01     Bob Jones  Widget B         1        50.0   
2           T003  2024-06-02   alice smith  Widget A         2        25.0   

  region  
0   east  
1   West  
2   EAST  


In [5]:
def transform(df):
    
    print("\n" + "=" * 50)
    print("TRANSFORM PHASE (Silver Layer)")
    print("=" * 50)
 
    rows_before = len(df)
 
    # --- Step 3a: Drop rows with missing critical fields ---
    print("\n--- Handling Missing Values ---")
    critical_columns = ["customer_name", "product"]
    missing_before = df[critical_columns].isnull().sum()
    print(f"Missing values in critical columns:\n{missing_before}")
 
    df = df.dropna(subset=critical_columns)
    print(f"Rows dropped due to nulls: {rows_before - len(df)}")
 
    # --- Step 3b: Remove exact duplicate rows ---
    print("\n--- Removing Duplicates ---")
    duplicates_count = df.duplicated().sum()
    print(f"Duplicate rows found: {duplicates_count}")
 
    df = df.drop_duplicates()
    print(f"Rows after removing duplicates: {len(df)}")
 
    # --- Step 3c: Validate quantity (must be > 0) ---
    print("\n--- Validating Quantity ---")
    invalid_qty = df[df["quantity"] <= 0]
    print(f"Rows with invalid quantity (<= 0): {len(invalid_qty)}")
    if len(invalid_qty) > 0:
        print(f"  Invalid rows:\n{invalid_qty[['transaction_id', 'quantity']]}")
 
    df = df[df["quantity"] > 0]
    print(f"Rows after quantity validation: {len(df)}")
 
    # --- Step 3d: Standardize text fields ---
    print("\n--- Standardizing Text Fields ---")
    df["customer_name"] = df["customer_name"].str.strip().str.title()
    df["region"] = df["region"].str.strip().str.title()
    df["product"] = df["product"].str.strip()
 
    print(f"Unique regions after standardization: {sorted(df['region'].unique())}")
    print(f"Unique customers after standardization: {sorted(df['customer_name'].unique())}")
 
    # --- Step 3e: Add computed column ---
    print("\n--- Adding Computed Columns ---")
    df["total_amount"] = df["quantity"] * df["unit_price"]
    print(f"Added 'total_amount' column (quantity * unit_price)")
 
    # --- Summary ---
    rows_after = len(df)
    print(f"\n--- Transform Summary ---")
    print(f"Rows before: {rows_before}")
    print(f"Rows after: {rows_after}")
    print(f"Rows removed: {rows_before - rows_after}")
    print(f"Columns: {list(df.columns)}")
 
    return df

In [6]:
clean_df = transform(raw_df)
print("\nCleaned data:")
print(clean_df)


TRANSFORM PHASE (Silver Layer)

--- Handling Missing Values ---
Missing values in critical columns:
customer_name    1
product          1
dtype: int64
Rows dropped due to nulls: 2

--- Removing Duplicates ---
Duplicate rows found: 0
Rows after removing duplicates: 8

--- Validating Quantity ---
Rows with invalid quantity (<= 0): 2
  Invalid rows:
  transaction_id  quantity
4           T005        -1
7           T008         0
Rows after quantity validation: 6

--- Standardizing Text Fields ---
Unique regions after standardization: ['East', 'West']
Unique customers after standardization: ['Alice Smith', 'Bob Jones']

--- Adding Computed Columns ---
Added 'total_amount' column (quantity * unit_price)

--- Transform Summary ---
Rows before: 10
Rows after: 6
Rows removed: 4
Columns: ['transaction_id', 'date', 'customer_name', 'product', 'quantity', 'unit_price', 'region', 'total_amount']

Cleaned data:
  transaction_id        date customer_name   product  quantity  unit_price  \
0        

In [7]:
def load(df, silver_path, gold_path):

    print("\n" + "=" * 50)
    print("LOAD PHASE")
    print("=" * 50)
 
    # --- Save Silver (cleaned data) ---
    print("\n--- Silver Layer (Cleaned Data) ---")
    df.to_csv(silver_path, index=False)
    print(f"Saved {len(df)} rows to: {silver_path}")
 
    # --- Build Gold (aggregated analytics) ---
    print("\n--- Gold Layer (Aggregated Analytics) ---")
 
    gold_df = df.groupby(["region", "product"]).agg(
        total_quantity=("quantity", "sum"),
        total_revenue=("total_amount", "sum"),
        transaction_count=("transaction_id", "count"),
        avg_unit_price=("unit_price", "mean")
    ).reset_index()
 
    # Round numeric columns
    gold_df["total_revenue"] = gold_df["total_revenue"].round(2)
    gold_df["avg_unit_price"] = gold_df["avg_unit_price"].round(2)
 
    # Sort by total revenue descending
    gold_df = gold_df.sort_values("total_revenue", ascending=False)
 
    gold_df.to_csv(gold_path, index=False)
    print(f"Saved {len(gold_df)} aggregated rows to: {gold_path}")
 
    # --- Print Gold summary ---
    print(f"\nGold layer output:")
    print(gold_df.to_string(index=False))
 
    # --- Pipeline summary ---
    print(f"\n--- Load Summary ---")
    print(f"Silver output: {silver_path} ({len(df)} rows)")
    print(f"Gold output: {gold_path} ({len(gold_df)} rows)")
    print(f"Total revenue: {gold_df['total_revenue'].sum():,.2f}")

In [8]:
load(clean_df, "silver_transactions.csv", "gold_transactions.csv")


LOAD PHASE

--- Silver Layer (Cleaned Data) ---
Saved 6 rows to: silver_transactions.csv

--- Gold Layer (Aggregated Analytics) ---
Saved 2 aggregated rows to: gold_transactions.csv

Gold layer output:
region  product  total_quantity  total_revenue  transaction_count  avg_unit_price
  East Widget A               8          200.0                  3            25.0
  West Widget B               3          150.0                  3            50.0

--- Load Summary ---
Silver output: silver_transactions.csv (6 rows)
Gold output: gold_transactions.csv (2 rows)
Total revenue: 350.00


In [9]:
def run_etl_pipeline(input_file, silver_output, gold_output):
    
    print("*" * 60)
    print("  ETL PIPELINE START")
    print("*" * 60)
 
    # Phase 1: Extract (Bronze)
    raw_df = extract(input_file)
 
    # Phase 2: Transform (Silver)
    clean_df = transform(raw_df)
 
    # Phase 3: Load (Silver + Gold)
    load(clean_df, silver_output, gold_output)
 
    print("\n" + "*" * 60)
    print("  ETL PIPELINE COMPLETE")
    print("*" * 60)

In [10]:
run_etl_pipeline(
    input_file="raw_transactions.csv",
    silver_output="silver_transactions.csv",
    gold_output="gold_transactions.csv"
)

************************************************************
  ETL PIPELINE START
************************************************************
EXTRACT PHASE (Bronze Layer)
Source file: raw_transactions.csv
Rows extracted: 10
Columns: 7

Column names: ['transaction_id', 'date', 'customer_name', 'product', 'quantity', 'unit_price', 'region']

Data types:
transaction_id     object
date               object
customer_name      object
product            object
quantity            int64
unit_price        float64
region             object
dtype: object

Missing values:
transaction_id    0
date              0
customer_name     1
product           1
quantity          0
unit_price        0
region            0
dtype: int64

First 3 rows:
  transaction_id        date customer_name   product  quantity  unit_price  \
0           T001  2024-06-01   alice smith  Widget A         3        25.0   
1           T002  2024-06-01     Bob Jones  Widget B         1        50.0   
2           T003  2024-06-02  

In [11]:
# Verify Silver
print("=== SILVER OUTPUT ===")
silver = pd.read_csv("silver_transactions.csv")
print(silver)
print(f"Shape: {silver.shape}")
 
# Verify Gold
print("\n=== GOLD OUTPUT ===")
gold = pd.read_csv("gold_transactions.csv")
print(gold)
print(f"Shape: {gold.shape}")

=== SILVER OUTPUT ===
  transaction_id        date customer_name   product  quantity  unit_price  \
0           T001  2024-06-01   Alice Smith  Widget A         3        25.0   
1           T002  2024-06-01     Bob Jones  Widget B         1        50.0   
2           T003  2024-06-02   Alice Smith  Widget A         2        25.0   
3           T006  2024-06-03   Alice Smith  Widget A         3        25.0   
4           T009  2024-06-05     Bob Jones  Widget B         1        50.0   
5           T010  2024-06-05     Bob Jones  Widget B         1        50.0   

  region  total_amount  
0   East          75.0  
1   West          50.0  
2   East          50.0  
3   East          75.0  
4   West          50.0  
5   West          50.0  
Shape: (6, 8)

=== GOLD OUTPUT ===
  region   product  total_quantity  total_revenue  transaction_count  \
0   East  Widget A               8          200.0                  3   
1   West  Widget B               3          150.0                  3   

   a